In [70]:
#Working with langchain,vectorstore & splitters
import langchain
import langchain_core
import langchain_text_splitters
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,SentenceTransformersTokenTextSplitter
)


In [71]:
text = """langchain is a framework for developing llm appls powered by llms.
          It connects different components such as models, prompts, configuration settings """

In [72]:
text_splitter = RecursiveCharacterTextSplitter(
                chunk_size= 50,
                chunk_overlap= 10,
)

In [73]:
chunks = text_splitter.split_text(text)

In [74]:
for i in chunks:
  print(i)

langchain is a framework for developing llm appls
llm appls powered by llms.
It connects different components such
such as models, prompts, configuration settings


In [75]:
for i,chunk in enumerate(chunks):
  print(f"Chunk {i+1}: {chunk}")

Chunk 1: langchain is a framework for developing llm appls
Chunk 2: llm appls powered by llms.
Chunk 3: It connects different components such
Chunk 4: such as models, prompts, configuration settings


In [76]:
#!pip install langchain_community
from langchain_community.embeddings import HuggingFaceEmbeddings


In [77]:
model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model)

In [78]:
#!pip list --check
#!pip install faiss-cpu --if not done

In [79]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [80]:
text_splitter = RecursiveCharacterTextSplitter(
                chunk_size= 50,
                chunk_overlap= 10,
)

In [81]:
text = """langchain is a framework for developing llm appls powered by llms
          It connects different components such as models, prompts, configuration settings """

In [82]:
chunks2 = text_splitter.create_documents([text])

In [83]:
for i in chunks2:
  print(i)

page_content='langchain is a framework for developing llm appls'
page_content='llm appls powered by llms'
page_content='It connects different components such'
page_content='such as models, prompts, configuration settings'


In [84]:
vectorstore = FAISS.from_documents(chunks2, embeddings)

In [85]:
all_docs = list(vectorstore.docstore._dict.values())

In [86]:
for i, doc in enumerate(all_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
langchain is a framework for developing llm appls

Document 2:
llm appls powered by llms

Document 3:
It connects different components such

Document 4:
such as models, prompts, configuration settings



In [87]:
query = "blackholes"

In [88]:
query

'blackholes'

In [89]:
results = vectorstore.similarity_search(query, k=4)

In [90]:
#Retrieving
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
It connects different components such

Result 2:
langchain is a framework for developing llm appls

Result 3:
llm appls powered by llms

Result 4:
such as models, prompts, configuration settings


In [91]:
#building retriever
retriever=vectorstore.as_retriever()

In [92]:
#print(dir(retriever))

In [93]:
query_embedding = embeddings.embed_query("langchain")

In [94]:
similar_docs = vectorstore.similarity_search_by_vector(query_embedding, k=5)  # top 5 results

In [95]:
for doc in similar_docs:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


In [96]:
query_embedding = embeddings.embed_query("Fishing boat")

In [97]:
similar_docs = vectorstore.similarity_search_by_vector(query_embedding, k=2)  # top 5 results

In [98]:
for doc in similar_docs:
    print(doc.page_content)

It connects different components such
langchain is a framework for developing llm appls


In [141]:
#Option 1
#Apply a similarity threshold, After retrieving results, filter them:
query_embedding = embeddings.embed_query("Fishing boat")
results = vectorstore.similarity_search_with_score_by_vector(query_embedding, k=5)

In [142]:
for doc, score in results:
    print(score, doc.page_content)

1.9245224 It connects different components such
2.0228248 langchain is a framework for developing llm appls
2.059639 such as models, prompts, configuration settings
2.0717306 llm appls powered by llms


In [143]:
filtered = [doc for doc, score in results if score < 1.92]
for doc in filtered:
    print(doc.page_content)
#Shows nothing

In [144]:
#Apply a similarity threshold, After retrieving results, filter them:
query_embedding = embeddings.embed_query("Langchain")
results = vectorstore.similarity_search_with_score_by_vector(query_embedding, k=5)

In [145]:
for doc, score in results:
    print(score, doc.page_content)

0.93989617 langchain is a framework for developing llm appls
1.7672544 It connects different components such
1.8723807 llm appls powered by llms
1.9892982 such as models, prompts, configuration settings


In [146]:
filtered = [doc for doc, score in results if score < 2]
for doc in filtered:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


In [147]:
threshold = 1.4
filtered = [doc for doc, score in results if score < threshold]
for doc in filtered:
    print(doc.page_content)

langchain is a framework for developing llm appls


#### Vector search always returns results — relevance is OUR responsibility

In [149]:
#Using similarity search
#If not using scores to filter by relevance
#only when you just want top-k results
"""Internally calls embeddings.embed_query()
Returns only documents (no scores)"""
results_new = vectorstore.similarity_search("Langchain", k=5)

for doc in results_new:
    print(doc.page_content)

langchain is a framework for developing llm appls
It connects different components such
llm appls powered by llms
such as models, prompts, configuration settings


In [150]:
#Using similarity_search_with_score
"""
Handles embedding internally, Returns (doc, score) pairs, Score is usually distance (lower = better)
"""
results_new2 = vectorstore.similarity_search_with_score("Langchain", k=5)

for doc, score in results_new2:
    print(score, doc.page_content)

threshold = 1.4  # based on our earlier observation

filtered = [
    doc for doc, score in results
    if score < threshold
]

for doc in filtered:
    print(doc.page_content)

    

0.93989617 langchain is a framework for developing llm appls
1.7672544 It connects different components such
1.8723807 llm appls powered by llms
1.9892982 such as models, prompts, configuration settings
langchain is a framework for developing llm appls


In [152]:
#Using similarity_search_with_relevance_scores
"""
Here Scores are normalized → 0 to 1, Higher = better (unlike distance)
"""
results = vectorstore.similarity_search_with_relevance_scores(query, k=5)
threshold = 0.7
filtered = [doc for doc, score in results if score > threshold]

if not filtered:
    print("No relevant results found")
else:
    for doc in filtered:
        print(doc.page_content)

No relevant results found


C:\Users\Ajay\AppData\Local\Temp\ipykernel_7096\3294070310.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8a43b021-eefe-473e-b7de-ba1aeb6eb880', metadata={}, page_content='It connects different components such'), -0.3002818643036582), (Document(id='eab69d3f-416c-4270-bf44-b73854a9803e', metadata={}, page_content='langchain is a framework for developing llm appls'), -0.37241139078932295), (Document(id='e9863760-32b5-4040-884a-2fcfeb9e6228', metadata={}, page_content='llm appls powered by llms'), -0.40772294770241735), (Document(id='c839bb9e-e7b7-4451-9cca-30c141ab0739', metadata={}, page_content='such as models, prompts, configuration settings'), -0.5052841354606468)]
  results = vectorstore.similarity_search_with_relevance_scores(query, k=5)


In [ ]:
#So better to use similarity_search_with_score()